# MetaboAge → Knowledge Graph (KG) Builder


In [11]:
# ! wget https://www.metaboage.info/static/website/variation-data.xlsx
# ! wget https://www.metaboage.info/static/website/chemical-modeling.xlsx

---
## 0 · Configuration — edit ONLY these two lines

In [12]:
import os
import re
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# BASE_PATH : root folder containing all raw input data
# OUT_PATH  : folder where all processed CSVs will be saved
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/MetaboAge/"

# ── Derived input paths (do not edit below this line) ────────────────────────
# BTO tissue ontology: name → BTO ID
BTO_PATH         = os.path.join(BASE_PATH, "databases_for_mapping/bto/BTO_ALL_COMBINED.csv")
# PubChem CID-Synonym flat file (tab-separated, no header)
PUBCHEM_SYN_PATH = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/CID-Synonym-filtered")
# PubChem compound table with IUPAC names and SMILES (pickled DataFrame)
PUBCHEM_PKL_PATH = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/combined_df.pkl")
# MetaboAge main data file
METABOAGE_PATH   = os.path.join(BASE_PATH, "metaboage/variation-data.xlsx")

os.makedirs(OUT_PATH, exist_ok=True)

print("Paths configured successfully.")
print(f"  Input  root : {BASE_PATH}")
print(f"  Output dir  : {OUT_PATH}")

Paths configured successfully.
  Input  root : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
  Output dir  : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/MetaboAge/


---
## 1 · Load reference dictionaries

In [13]:
# ── 1a. BTO tissue ontology — name → BTO ID ───────────────────────────────────
BTO_file = pd.read_csv(BTO_PATH)
BTO_file_dict = dict(zip(BTO_file['name'], BTO_file['ID']))  # {tissue_name: BTO:XXXXXXX}
print(f"Loaded {len(BTO_file_dict):,} BTO tissue entries")

Loaded 6,608 BTO tissue entries


In [14]:
# ── 1b. PubChem synonym → CID lookup ─────────────────────────────────────────
# CID-Synonym-filtered: col-0 = CID, col-1 = synonym string.
Pubchem_Syn_fil = pd.read_csv(PUBCHEM_SYN_PATH, sep='\t', header=None)
Pubchem_Syn_fil_dict = dict(zip(Pubchem_Syn_fil[1], Pubchem_Syn_fil[0]))  # {synonym: CID}

# Pre-build lowercase version for case-insensitive lookups
Pubchem_Syn_fil_dict_lower = {str(k).lower(): v for k, v in Pubchem_Syn_fil_dict.items()}
print(f"Loaded {len(Pubchem_Syn_fil_dict):,} PubChem synonyms")

Loaded 102,877,691 PubChem synonyms


In [6]:
# ── 1c. PubChem CID → IUPAC name and SMILES ──────────────────────────────────
Pubchem = pd.read_pickle(PUBCHEM_PKL_PATH)
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))  # {CID: IUPAC}
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))       # {CID: SMILES}
print(f"Loaded {len(Pubchem_IUPAC_CID_Dict):,} CID → IUPAC mappings")
Pubchem.head(3)

Loaded 119,177,440 CID → IUPAC mappings


,PUBCHEM_COMPOUND_CID,PUBCHEM_IUPAC_NAME,PUBCHEM_SMILES
0,56500001,"2-[1-(4,6-dimethylpyrimidin-2-yl)-3,5-dimethyl...",CC1=CC(=NC(=N1)N2C(=C(C(=N2)C)CC(=O)NC3=C(N=CC...
1,56500002,N-[2-(2-methoxyethoxy)pyridin-3-yl]-2-[[4-(4-m...,COCCOC1=C(C=CC=N1)NC(=O)CSC2=NN=CN2C3=CC=C(C=C...
2,56500003,N-[2-(2-methoxyethoxy)pyridin-3-yl]-1-phenyl-5...,COCCOC1=C(C=CC=N1)NC(=O)C2=NN(C(=N2)C3=CC=CS3)...


---
## 2 · Load and prepare MetaboAge base DataFrame

The raw file is loaded **once** here and shared by all three modules below.  
All column renames and PubChem lookups are applied in this single block.


In [15]:
# ── 2a. Load raw MetaboAge data ───────────────────────────────────────────────
MetaboAge_raw = pd.read_excel(METABOAGE_PATH)
print(f"Loaded {len(MetaboAge_raw):,} rows")
print("Columns:", list(MetaboAge_raw.columns))
MetaboAge_raw.head(3)

Loaded 1,519 rows
Columns: ['Metabolite', 'Method', 'Mean value/beta value', 'Standard deviation (+-)/standard error', 'Unit of measurment', '(NEW) Age group', 'Gender', 'Tipe of sample used in study', 'Study  PMID reference']


,Metabolite,Method,Mean value/beta value,Standard deviation (+-)/standard error,Unit of measurment,(NEW) Age group,Gender,Tipe of sample used in study,Study PMID reference
0,kynurenic acid,HPLC,2.84,NaN,fmol/microL,"35.4 ± 2.2, ranging from 25\nto 50 years",both genders,cerebrospinal fluid,PMID:16088227
1,kynurenic acid,HPLC,4.09,NaN,fmol/microL,"61.6 ± 3.3 years, ranging from 50 to 74",both genders,cerebrospinal fluid,PMID:16088227
2,linoleic acid,GC,1,NaN,%,[21-33],both genders,skin,PMID:20514327


In [16]:
# ── 2b. Coerce numeric and inspect range ──────────────────────────────────────
MetaboAge_raw['Mean value/beta value'] = pd.to_numeric(MetaboAge_raw['Mean value/beta value'], errors='coerce')
print(f"Mean value/beta value range: {MetaboAge_raw['Mean value/beta value'].min():.4f} → {MetaboAge_raw['Mean value/beta value'].max():.4f}")

Mean value/beta value range: -9578.0000 → 4644733.0000


In [17]:
# ── 2c. Resolve metabolite name → PubChem CID (case-insensitive) ──────────────
MetaboAge_raw['Head'] = MetaboAge_raw['Metabolite'].str.lower().map(Pubchem_Syn_fil_dict_lower)
# Strip float suffix (e.g. 12345.0 → 12345)
MetaboAge_raw['Head'] = MetaboAge_raw['Head'].astype(str).str.replace(r'\.0$', '', regex=True)

# Annotate IUPAC name and SMILES for each resolved CID
# Original created 'Head_SMILE_name' but desired_cols selected 'Head_SMILES_name' → dropped silently.
MetaboAge_raw['Head_detail_name'] = MetaboAge_raw['Head'].map(Pubchem_IUPAC_CID_Dict)
MetaboAge_raw['Head_SMILE_name']  = MetaboAge_raw['Head'].map(Pubchem_CID_Smile_Dict)

# Map tissue/biofluid name → BTO ID
# Note: source column has a typo 'Tipe' (not 'Type') — preserved as-is to match source
MetaboAge_raw['Tail'] = MetaboAge_raw['Tipe of sample used in study'].map(BTO_file_dict)

# Rename source columns to KG schema names
MetaboAge_raw = MetaboAge_raw.rename(columns={
    'Tipe of sample used in study': 'Tail_detail_name',  # tissue name → Tail_detail_name
    'Metabolite':                   'Head_Alt_name'       # original metabolite name
})

print(f"PubChem CID resolved for {(MetaboAge_raw['Head'] != 'nan').sum():,} / {len(MetaboAge_raw):,} rows")
print(f"BTO tissue resolved for  {MetaboAge_raw['Tail'].notna().sum():,} / {len(MetaboAge_raw):,} rows")
MetaboAge_raw.head(3)

PubChem CID resolved for 1,157 / 1,519 rows
BTO tissue resolved for  1,110 / 1,519 rows


,Head_Alt_name,Method,Mean value/beta value,Standard deviation (+-)/standard error,Unit of measurment,(NEW) Age group,Gender,Tail_detail_name,Study PMID reference,Head,Head_detail_name,Head_SMILE_name,Tail
0,kynurenic acid,HPLC,2.84,NaN,fmol/microL,"35.4 ± 2.2, ranging from 25\nto 50 years",both genders,cerebrospinal fluid,PMID:16088227,3845,4-oxo-1H-quinoline-2-carboxylic acid,C1=CC=C2C(=C1)C(=O)C=C(N2)C(=O)O,BTO:0000237
1,kynurenic acid,HPLC,4.09,NaN,fmol/microL,"61.6 ± 3.3 years, ranging from 50 to 74",both genders,cerebrospinal fluid,PMID:16088227,3845,4-oxo-1H-quinoline-2-carboxylic acid,C1=CC=C2C(=C1)C(=O)C=C(N2)C(=O)O,BTO:0000237
2,linoleic acid,GC,1.00,NaN,%,[21-33],both genders,skin,PMID:20514327,5280450,"(9Z,12Z)-octadeca-9,12-dienoic acid",CCCCC/C=C\C/C=C\CCCCCCCC(=O)O,BTO:0001253


---
## 3 · Module 1 — Chemical → Tissue edges

Each metabolite–tissue co-occurrence is one edge.  
Rows where the tissue could not be mapped to a BTO ID are dropped.  
Duplicates on `(Head, Relation, Tail)` are deduplicated before saving.

In [19]:
# ── 3a. Build Chemical–Tissue edge table ──────────────────────────────────────
MetaboAge_CT = MetaboAge_raw.copy()

MetaboAge_CT['Head_type']       = 'ChemicalEntity'
MetaboAge_CT['Tail_type']       = 'Tissue'
MetaboAge_CT['Relation']        = 'ChemicalEntity_Tissue'
MetaboAge_CT['Head_ID_IS']      = 'Pubchem'
MetaboAge_CT['Tail_ID_IS']      = 'BTO_ID'
MetaboAge_CT['Species']         = 'Homo sapiens'
MetaboAge_CT['Relation_Source'] = 'MetaboAge'

# Select only columns present in the DataFrame (safe schema)
desired_cols = [
    'Head', 'Relation', 'Tail', 'Head_type', 'Relation_type', 'Tail_type',
    'Relation_Source', 'KG_Source', 'Species',
    'Head_Alt_name', 'Head_detail_name', 'Head_SMILE_name',
    'Tail_detail_name', 'Head_ID_IS', 'Tail_ID_IS',
    'Study PMID reference'
]
MetaboAge_Chem_Tissue = MetaboAge_CT[[c for c in desired_cols if c in MetaboAge_CT.columns]]

# Drop rows where BTO tissue ID could not be resolved
before = len(MetaboAge_Chem_Tissue)
MetaboAge_Chem_Tissue = MetaboAge_Chem_Tissue[~MetaboAge_Chem_Tissue['Tail'].isna()]
print(f"BTO filter: {len(MetaboAge_Chem_Tissue):,} kept / {before - len(MetaboAge_Chem_Tissue):,} dropped")

# Deduplicate on the triple (Head, Relation, Tail)
before = len(MetaboAge_Chem_Tissue)
MetaboAge_Chem_Tissue = MetaboAge_Chem_Tissue.drop_duplicates(subset=['Head', 'Relation', 'Tail'])
print(f"After dedup: {len(MetaboAge_Chem_Tissue):,} unique triples (removed {before - len(MetaboAge_Chem_Tissue):,} duplicates)")
MetaboAge_Chem_Tissue.head(3)

BTO filter: 1,110 kept / 409 dropped
After dedup: 264 unique triples (removed 846 duplicates)


,Head,Relation,Tail,Head_type,Tail_type,Relation_Source,Species,Head_Alt_name,Head_detail_name,Head_SMILE_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,3845,ChemicalEntity_Tissue,BTO:0000237,ChemicalEntity,Tissue,MetaboAge,Homo sapiens,kynurenic acid,4-oxo-1H-quinoline-2-carboxylic acid,C1=CC=C2C(=C1)C(=O)C=C(N2)C(=O)O,cerebrospinal fluid,Pubchem,BTO_ID
2,5280450,ChemicalEntity_Tissue,BTO:0001253,ChemicalEntity,Tissue,MetaboAge,Homo sapiens,linoleic acid,"(9Z,12Z)-octadeca-9,12-dienoic acid",CCCCC/C=C\C/C=C\CCCCCCCC(=O)O,skin,Pubchem,BTO_ID
4,124886,ChemicalEntity_Tissue,BTO:0000089,ChemicalEntity,Tissue,MetaboAge,Homo sapiens,glutathione,(2S)-2-amino-5-[[(2R)-1-(carboxymethylamino)-1...,C(CC(=O)N[C@@H](CS)C(=O)NCC(=O)O)[C@@H](C(=O)O)N,blood,Pubchem,BTO_ID


In [20]:
# Corrected to 'MetaboAge_Human_Chem_Tissue.csv'.
out_path = os.path.join(OUT_PATH, 'MetaboAge_Human_Chem_Tissue.csv')
MetaboAge_Chem_Tissue.to_csv(out_path, index=False)
print(f"Saved {len(MetaboAge_Chem_Tissue):,} rows → {out_path}")

Saved 264 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data/MetaboAge/MetaboAge_Human_Chem_Tissue.csv


---
## 4 · Module 2 — Chemical → BiologicalProcess (direct, no trend fitting)


In [21]:
# ── 4a. Build flat Chemical–BiologicalProcess edge table ──────────────────────
# Reuse the prepared MetaboAge_raw DataFrame — no need to reload the file.
MetaboAge_CB = MetaboAge_raw.copy()

# Override the Tail column: all edges point to the aging GO term
MetaboAge_CB['Head_ID_IS']      = 'Pubchem'
MetaboAge_CB['Tail_ID_IS']      = 'Quick_GO'
MetaboAge_CB['Tail']            = 'GO:0007568'
MetaboAge_CB['Tail_detail_name']= 'obsolete aging'
MetaboAge_CB['Tail_type']       = 'BiologicalProcess'
MetaboAge_CB['Head_type']       = 'ChemicalEntity'
MetaboAge_CB['Relation']        = 'ChemicalEntity_BiologicalProcess'
MetaboAge_CB['Species']         = 'Homo sapiens'
MetaboAge_CB['Relation_Source'] = 'MetaboAge'

desired_cols = [
    'Head', 'Relation', 'Tail', 'Head_type', 'Relation_type', 'Tail_type',
    'Relation_Source', 'KG_Source', 'Species',
    'Head_Alt_name', 'Head_detail_name', 'Head_SMILE_name',
    'Tail_detail_name', 'Head_ID_IS', 'Tail_ID_IS',
    'Study PMID reference', 'Slope', 'Trend'
]
MetaboAge_Chem_BioPro_direct = MetaboAge_CB[[c for c in desired_cols if c in MetaboAge_CB.columns]]

print(f"Direct Chemical–BiologicalProcess edges: {len(MetaboAge_Chem_BioPro_direct):,} rows")
MetaboAge_Chem_BioPro_direct.head(3)

Direct Chemical–BiologicalProcess edges: 1,519 rows


,Head,Relation,Tail,Head_type,Tail_type,Relation_Source,Species,Head_Alt_name,Head_detail_name,Head_SMILE_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,3845,ChemicalEntity_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,MetaboAge,Homo sapiens,kynurenic acid,4-oxo-1H-quinoline-2-carboxylic acid,C1=CC=C2C(=C1)C(=O)C=C(N2)C(=O)O,obsolete aging,Pubchem,Quick_GO
1,3845,ChemicalEntity_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,MetaboAge,Homo sapiens,kynurenic acid,4-oxo-1H-quinoline-2-carboxylic acid,C1=CC=C2C(=C1)C(=O)C=C(N2)C(=O)O,obsolete aging,Pubchem,Quick_GO
2,5280450,ChemicalEntity_BiologicalProcess,GO:0007568,ChemicalEntity,BiologicalProcess,MetaboAge,Homo sapiens,linoleic acid,"(9Z,12Z)-octadeca-9,12-dienoic acid",CCCCC/C=C\C/C=C\CCCCCCCC(=O)O,obsolete aging,Pubchem,Quick_GO


In [22]:
# to avoid being overwritten by Module 3 which saved to the same filename.
out_path = os.path.join(OUT_PATH, 'MetaboAge_Human_Chem_BioPro_direct.csv')
MetaboAge_Chem_BioPro_direct.to_csv(out_path, index=False)
print(f"Saved {len(MetaboAge_Chem_BioPro_direct):,} rows → {out_path}")

Saved 1,519 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data/MetaboAge/MetaboAge_Human_Chem_BioPro_direct.csv
